In [ ]:
# ==========================================
# 1. استيراد المكتبات الأساسية
# ==========================================
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# ==========================================
# 2. الإعدادات العامة (CONFIG)
# ==========================================
CONFIG = {
    "dataset_path": "Custom*CEDAR Handwriting Recognition*",
    "img_size": (224, 224),
    "batch_size": 32,
    "epochs": 40,
    "learning_rate": 1e-4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42
}

# تثبيت العشوائية لضمان نفس النتائج كل مرة
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])

# ==========================================
# 3. كلاس تجهيز البيانات (SignatureDataset)
# ==========================================
class SignatureDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.genuine_path = os.path.join(root_dir, 'full_org')
        self.forged_path = os.path.join(root_dir, 'full_forg')
        
        self.writers = {}
        all_genuine = sorted([f for f in os.listdir(self.genuine_path) if f.endswith('.png')])
        all_forged = sorted([f for f in os.listdir(self.forged_path) if f.endswith('.png')])
        
        for f in all_genuine:
            try:
                wid = int(f.split('_')[1])
                if wid not in self.writers: self.writers[wid] = {'gen': [], 'forg': []}
                self.writers[wid]['gen'].append(os.path.join(self.genuine_path, f))
            except: continue
            
        for f in all_forged:
            try:
                wid = int(f.split('_')[1])
                if wid in self.writers: self.writers[wid]['forg'].append(os.path.join(self.forged_path, f))
            except: continue
        
        writer_ids = list(self.writers.keys())
        train_ids, test_ids = train_test_split(writer_ids, test_size=0.2, random_state=CONFIG['seed'])
        active_ids = train_ids if split == 'train' else test_ids
        
        self.pairs = []
        for wid in active_ids:
            gens = self.writers[wid]['gen']
            forgs = self.writers[wid]['forg']
            # أزواج متشابهة (0)
            for i in range(len(gens)):
                for j in range(i + 1, len(gens)):
                    self.pairs.append((gens[i], gens[j], 0))
            # أزواج مختلفة (تزوير) (1)
            for g in gens:
                for f in forgs:
                    self.pairs.append((g, f, 1))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        path1, path2, label = self.pairs[idx]
        img1 = Image.open(path1).convert("RGB")
        img2 = Image.open(path2).convert("RGB")
        if self.transform:
            img1, img2 = self.transform(img1), self.transform(img2)
        return img1, img2, torch.tensor(label, dtype=torch.float32)

# ==========================================
# 4. بناء الموديل (Siamese Transformer)
# ==========================================
class SiameseTransformer(nn.Module):
    def __init__(self):
        super(SiameseTransformer, self).__init__()
        # Backbone: EfficientNet-B0
        efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.backbone = efficientnet.features
        
        # Transformer Layer
        self.feature_dim = 1280
        self.pos_embedding = nn.Parameter(torch.randn(1, 49, self.feature_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=self.feature_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Head
        self.fc = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128)
        )

    def forward_one(self, x):
        x = self.backbone(x) # Output: [B, 1280, 7, 7]
        x = x.view(x.size(0), self.feature_dim, -1).permute(0, 2, 1) # [B, 49, 1280]
        x = self.transformer(x + self.pos_embedding)
        x = torch.mean(x, dim=1)
        return self.fc(x)

    def forward(self, img1, img2):
        return self.forward_one(img1), self.forward_one(img2)

# ==========================================
# 5. تجهيز التحميل والتدريب
# ==========================================
transform = transforms.Compose([
    transforms.Resize(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = SignatureDataset(CONFIG['dataset_path'], split='train', transform=transform)
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)

model = SiameseTransformer().to(CONFIG['device'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
criterion = nn.CosineEmbeddingLoss(margin=0.5)

# ==========================================
# 6. حلقة التدريب (The Training Loop)
# ==========================================
print(f"الموديل بدأ التدريب على: {CONFIG['device']}")

for epoch in range(CONFIG['epochs']):
    model.train()
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
    
    for img1, img2, labels in loop:
        img1, img2, labels = img1.to(CONFIG['device']), img2.to(CONFIG['device']), labels.to(CONFIG['device'])
        # تحويل الـ labels: 0 يبقى 1 (متشابه)، و 1 يبقى -1 (مختلف)
        target = 1 - 2 * labels 
        
        optimizer.zero_grad()
        out1, out2 = model(img1, img2)
        loss = criterion(out1, out2, target)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    print(f"Epoch {epoch+1} finished. Average Loss: {epoch_loss/len(train_loader):.4f}")

# حفظ الموديل النهائي
torch.save(model.state_dict(), "final_signature_model.pth")
print("تم الانتهاء من التدريب وحفظ الموديل بنجاح!")

In [ ]:
# السطرين دول هما اللي "بينادوا" على الدالة عشان تطلع النتيجة
orig_img = "/kaggle/input/cedardataset/signatures/full_org/original_1_1.png"
test_img = "/kaggle/input/cedardataset/signatures/full_forg/forgeries_1_1.png"

# مناداة الدالة اللي إنت كتبتها فوق
show_report(orig_img, test_img)

# كود إضافي عشان يحسب لك الأكيروسي (الدقة) الكلية للمشروع
print("\n🔍 جاري حساب الدقة الكلية (Accuracy)...")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for i, (img1, img2, label) in enumerate(train_loader):
        if i > 30: break  # بنجرب على 30 عينة بسرعة عشان ما يطولش
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        o1, o2 = model(img1, img2)
        dist = torch.nn.functional.pairwise_distance(o1, o2)
        preds = (dist < 0.5).float()
        correct += (preds == label).sum().item()
        total += label.size(0)

print(f"🎯 الأكيروسي الكلية للموديل هي: {(correct/total)*100:.2f}%")